In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn umap-learn joblib tqdm psutil

# Proyecto de Reducción de Dimensionalidad y Clasificación de Imágenes de Moda


## Objetivo
Este proyecto explora y compara diferentes técnicas de reducción de dimensionalidad sobre el dataset Fashion MNIST, buscando transformar el espacio de 784 características (píxeles) en un espacio más compacto que mantenga la información esencial para la clasificación.

### Aspectos Clave
- **SEED = 42** para reproducibilidad
- Se evaluarán múltiples estrategias de normalización
- No se asume una única normalización por defecto (e.g., /255)
- Todo el proceso sigue una metodología sistemática y reproducible

### Estructura del Notebook
1. **Bloque A**: Preparación, normalización y análisis exploratorio
2. **Bloque B**: Reducción de dimensionalidad y análisis de estabilidad
3. **Bloque C**: Clasificación, evaluación y selección multi-criterio

### Dataset
- 70,000 imágenes en escala de grises (28×28 píxeles)
- 10 clases de prendas de vestir
- 784 características (dimensionalidad original)
- División: 80% entrenamiento, 20% prueba

In [ ]:
# Importaciones y configuración del entorno
import os
import sys
import json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Importaciones locales
sys.path.append('..')
from src.data.loader import ejecutar_pipeline_carga
from src.data.normalizers import obtener_normalizador
from src.utils.helpers import (
    guardar_figura, vision_general_dataset, cuadricula_muestras_eda,
    histograma_pixeles_eda, imagenes_promedio_eda
)

# Configuración global
SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8')

# Guardar versiones en requirements.txt
import pkg_resources
packages = [str(pkg) for pkg in pkg_resources.working_set]
with open('../requirements.txt', 'w') as f:
    f.write('\n'.join(packages))

print("Versiones principales:")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {pkg_resources.get_distribution('scikit-learn').version}")
print(f"Matplotlib: {plt.__version__}")

OSError: 'seaborn' is not a valid package style, path of style file, URL of style file, or library style name (library styles are listed in `style.available`)

In [ ]:
# Crear estructura de directorios
directorios = [
    'data/raw',
    'data/processed',
    'data/processed/normalizations',
    'data/processed/embeddings',
    'outputs/figures',
    'outputs/tables',
    'outputs/models'
]

for d in directorios:
    os.makedirs(f'../{d}', exist_ok=True)
print("Estructura de directorios creada:") 
for d in directorios:
    print(f"- {d}")

# BLOQUE A: Preparación, Normalización y Análisis Exploratorio

## Objetivo
Este bloque se centra en:
1. La carga y validación inicial de los datos
2. El análisis exploratorio detallado
3. La evaluación y selección de normalizadores

### Flujo del Bloque
1. Carga de datos crudos
2. Validación básica
3. Análisis exploratorio detallado
4. Evaluación de normalizadores
5. Generación de datasets normalizados
6. Decisión operativa sobre dimensiones candidatas

In [ ]:
# Ejecutar pipeline de carga
resultado = ejecutar_pipeline_carga(
    descarga=True,
    dir_crudo="../data/raw/",
    dir_procesado="../data/processed/",
    sobreescribir=False
)

print("Rutas generadas:")
for k, v in resultado['rutas'].items():
    print(f"{k}: {v}")

print("\nMetadatos:")
print(json.dumps(resultado['meta'], indent=2))

In [ ]:
# Cargar datos procesados baseline y realizar validaciones
X_train = np.load("../data/processed/baseline/X_train.npy")
X_test = np.load("../data/processed/baseline/X_test.npy")
y_train = pd.read_csv("../data/processed/baseline/y_train.csv")
y_test = pd.read_csv("../data/processed/baseline/y_test.csv")

# Validaciones básicas
assert X_train.ndim == 2, "X_train debe ser 2D"
assert X_test.ndim == 2, "X_test debe ser 2D"
assert X_train.shape[1] == X_test.shape[1], "Número de características debe coincidir"
assert len(X_train) == len(y_train), "Número de muestras debe coincidir con etiquetas (train)"
assert len(X_test) == len(y_test), "Número de muestras debe coincidir con etiquetas (test)"
assert X_train.dtype == X_test.dtype, "Tipos de datos deben coincidir"
assert 0 <= X_train.min() <= X_train.max() <= 1, "Valores deben estar en [0,1]"
assert 0 <= X_test.min() <= X_test.max() <= 1, "Valores deben estar en [0,1]"

print("Shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

print("\nRangos:")
print(f"X_train: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"X_test: [{X_test.min():.3f}, {X_test.max():.3f}]")

# Guardar visión general
vision = vision_general_dataset(
    X_train, X_test, 
    y_train['etiqueta'].values, y_test['etiqueta'].values,
    csv_salida="../outputs/tables/vision_general_dataset.csv"
)
print("\nVisión general guardada en outputs/tables/vision_general_dataset.csv")

## Análisis Exploratorio de Datos (EDA)

### A. Estadísticas Básicas y Distribución de Clases
Analizamos la distribución de clases y las características básicas de nuestros datos.

In [ ]:
# Definir nombres de clase
nombres_clases = {
    0: "Camiseta/Top",
    1: "Pantalón",
    2: "Suéter",
    3: "Vestido",
    4: "Abrigo",
    5: "Sandalia",
    6: "Camisa",
    7: "Zapatilla",
    8: "Bolso",
    9: "Botín"
}

# Crear DataFrame con conteo de clases
y_train_valores = y_train['etiqueta'].values
df_conteo = pd.DataFrame({
    'etiqueta': range(10),
    'conteo': np.bincount(y_train_valores),
    'nombre_clase': [nombres_clases[i] for i in range(10)]
})

# Guardar conteos
df_conteo.to_csv("../outputs/tables/class_counts.csv", index=False)

# Visualizar distribución
fig, ax = plt.subplots(figsize=(12,6))
sns.barplot(data=df_conteo, x='nombre_clase', y='conteo', ax=ax)
ax.set_title('Distribución de Clases (Conjunto de Entrenamiento)', fontsize=12)
ax.set_xlabel('Clase', fontsize=10)
ax.set_ylabel('Conteo', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
guardar_figura(fig, "../outputs/figures/EDA_class_dist")

### B. Visualización de Muestras
Generamos una cuadrícula de muestras aleatorias y las imágenes promedio por clase para entender mejor nuestros datos.

In [ ]:
# Generar cuadrícula de muestras aleatorias
fig = cuadricula_muestras_eda(
    X_train, y_train_valores,
    n_columnas=10, n_filas=5,
    forma_img=(28,28),
    generador=np.random.default_rng(SEED),
    ruta_guardado="../outputs/figures/EDA_sample_grid",
    nombres_clases=nombres_clases
)

# Generar y guardar imágenes promedio por clase
rutas_promedio = imagenes_promedio_eda(
    X_train, y_train_valores,
    forma_img=(28,28),
    directorio_salida="../outputs/figures/EDA_mean_images",
    nombres_clases=nombres_clases
)

print("Visualizaciones guardadas en:")
print("- outputs/figures/EDA_sample_grid")
print("- outputs/figures/EDA_mean_images")

### C. Estadísticas de Píxeles
Analizamos la distribución de valores de píxeles para entender mejor nuestros datos a nivel de característica.

In [ ]:
# Histograma de intensidades de píxeles
fig_hist = histograma_pixeles_eda(X_train, bins=50, ruta_guardado="../outputs/figures/EDA_pixel_hist")

# Boxplot de distribución por característica (primeras 1000 muestras para visualización)
fig, ax = plt.subplots(figsize=(15,5))
sns.boxplot(data=pd.DataFrame(X_train[:1000]), ax=ax)
ax.set_title('Distribución de Valores de Píxeles (primeras 1000 muestras)')
ax.set_xlabel('Índice del Píxel')
ax.set_ylabel('Valor')
guardar_figura(fig, "../outputs/figures/EDA_pixel_boxplot")

# Calcular estadísticas descriptivas por píxel
stats_pixeles = pd.DataFrame({
    'media': X_train.mean(axis=0),
    'std': X_train.std(axis=0),
    'min': X_train.min(axis=0),
    'max': X_train.max(axis=0),
    'zeros': (X_train == 0).sum(axis=0) / len(X_train)
})

stats_pixeles.to_csv("../outputs/tables/pixel_stats.csv")
print("\nEstadísticas de píxeles guardadas en outputs/tables/pixel_stats.csv")

### D. Análisis de Dimensionalidad
Exploramos las características del espacio de alta dimensión mediante PCA, información mutua y estimación de dimensión intrínseca.

In [ ]:
# PCA exploratorio
from src.utils.helpers import calcular_pca_varianza_acumulada, calcular_informacion_mutua_componentes
from src.utils.helpers import estimar_dimension_intrinseca

# Calcular varianza acumulada PCA
var_acum = calcular_pca_varianza_acumulada(X_train, max_componentes=200)
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(np.arange(1, len(var_acum)+1), var_acum)
ax.set_title('Varianza Explicada Acumulada (PCA)')
ax.set_xlabel('Número de Componentes')
ax.set_ylabel('Varianza Explicada Acumulada')
ax.axhline(y=0.9, color='r', linestyle='--', label='90%')
ax.axhline(y=0.95, color='g', linestyle='--', label='95%')
ax.grid(True)
ax.legend()
guardar_figura(fig, "../outputs/figures/EDA_pca_cumvar")

# Calcular números de componentes para umbrales
n_90 = np.searchsorted(var_acum, 0.9) + 1
n_95 = np.searchsorted(var_acum, 0.95) + 1

print(f"Componentes necesarios para 90% de varianza: {n_90}")
print(f"Componentes necesarios para 95% de varianza: {n_95}")

# Información mutua con las etiquetas
df_mi = calcular_informacion_mutua_componentes(
    X_train, y_train_valores,
    n_componentes=100,
    semilla=SEED
)
df_mi.to_csv("../outputs/tables/mi_components.csv", index=False)

# Estimación de dimensión intrínseca
est_dim = estimar_dimension_intrinseca(X_train, max_componentes=200)
with open("../outputs/tables/intrinsic_dim_estimates.json", 'w') as f:
    json.dump(est_dim, f, indent=2)

print("\nEstimaciones de dimensión intrínseca:")
print(json.dumps(est_dim, indent=2))

## Evaluación de Normalizadores

Implementamos y evaluamos diferentes técnicas de normalización para seleccionar las más apropiadas para nuestros datos:

1. MinMax [0,1] (baseline; requerido para NMF)
2. Standard (z-score): mean 0, SD 1 
3. RobustScaler: centrado por mediana / escala por IQR
4. Per-image standardization
5. L2-normalization (unit norm)
6. PCA whitening
7. CLAHE (mejora de contraste)

In [ ]:
# Lista de normalizadores a evaluar
normalizadores = ['minmax', 'standard', 'robust', 'per_image', 'l2', 'pca_whiten', 'clahe']

# Tamaño de submuestra para evaluación rápida
n_submuestra = 5000

# Crear submuestra estratificada
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=None, train_size=n_submuestra, random_state=SEED)
idx_train, _ = next(sss.split(X_train, y_train_valores))

X_train_sub = X_train[idx_train]
y_train_sub = y_train_valores[idx_train]

# Evaluación comparativa
from src.utils.helpers import evaluar_normalizadores
df_eval = evaluar_normalizadores(
    X_train_sub, X_test,
    y_train_sub, y_test['etiqueta'].values,
    lista_normalizadores=normalizadores,
    csv_salida="../outputs/tables/normalizer_comparison.csv",
    fig_salida="../outputs/figures/normalizer_comparison"
)

print("Resultados de evaluación de normalizadores:")
print(df_eval.to_string())

# Elegir mejores normalizadores
from src.utils.helpers import elegir_normalizadores_y_guardar
meta_norm = elegir_normalizadores_y_guardar(
    df_eval,
    top_k=2,
    meta_salida="../data/processed/normalizadores_elegidos.json",
    preferir_no_negativo='minmax'
)

print("\nNormalizadores elegidos:")
print(json.dumps(meta_norm, indent=2))

## Generación de Datasets Normalizados

Generamos y guardamos versiones normalizadas del dataset usando los normalizadores seleccionados. Además, aseguramos tener una versión con MinMax para NMF.

In [ ]:
# Generar versiones normalizadas para cada normalizador elegido
from src.data.normalizers import obtener_normalizador
from src.data.loader import guardar_arrays_procesados

for norm in meta_norm['elegidos']:
    print(f"\nProcesando normalizador: {norm}")
    
    # Obtener y ajustar normalizador
    normalizador = obtener_normalizador(norm)
    X_train_norm = normalizador.fit_transform(X_train)
    X_test_norm = normalizador.transform(X_test)
    
    # Guardar versión normalizada
    dir_salida = f"../data/processed/normalizations/{norm}"
    os.makedirs(dir_salida, exist_ok=True)
    
    guardar_arrays_procesados(
        X_train_norm, X_test_norm,
        y_train, y_test,
        dir_salida=dir_salida,
        nombre_normalizador=norm,
        meta={
            "marca_tiempo": datetime.utcnow().isoformat() + "Z",
            "parametros": str(normalizador.get_params()),
            "forma_entrada": X_train.shape
        }
    )
    print(f"Guardado en {dir_salida}")

# Asegurar versión MinMax para NMF si no está ya incluida
if 'minmax' not in meta_norm['elegidos']:
    print("\nGenerando versión MinMax para NMF...")
    normalizador = obtener_normalizador('minmax')
    X_train_norm = normalizador.fit_transform(X_train)
    X_test_norm = normalizador.transform(X_test)
    
    dir_salida = "../data/processed/normalizations/minmax"
    os.makedirs(dir_salida, exist_ok=True)
    
    guardar_arrays_procesados(
        X_train_norm, X_test_norm,
        y_train, y_test,
        dir_salida=dir_salida,
        nombre_normalizador='minmax',
        meta={
            "marca_tiempo": datetime.utcnow().isoformat() + "Z",
            "nota": "Generado específicamente para NMF",
            "parametros": str(normalizador.get_params()),
            "forma_entrada": X_train.shape
        }
    )
    print(f"Guardado en {dir_salida}")

## Decisión Operativa: Dimensiones Candidatas

Combinamos los resultados de:
1. PCA (varianza acumulada)
2. Información Mutua
3. Estimación de dimensión intrínseca

Para determinar el conjunto de dimensiones candidatas para la reducción.

In [ ]:
# Calcular dimensiones candidatas
from src.utils.helpers import calcular_candidatos_dimension

# Información de PCA
info_pca = {
    'n_en_90': n_90,
    'n_en_95': n_95
}

# Calcular candidatos y guardar
candidatos = calcular_candidatos_dimension(
    X_train,
    info_pca=info_pca,
    df_mi=df_mi,
    est_intrinseca=est_dim,
    csv_salida="../outputs/tables/candidatos_dimension.csv"
)

print("Dimensiones candidatas:")
print(candidatos)

# Guardar decisiones operativas
decisiones = {
    "normalizadores_elegidos": meta_norm['elegidos'],
    "dimensiones_candidatas": candidatos,
    "marca_tiempo": datetime.utcnow().isoformat() + "Z",
    "notas": {
        "pca_90": n_90,
        "pca_95": n_95,
        "dimension_intrinseca": est_dim['dimension_estimada']
    }
}

with open("../outputs/tables/decisiones_operativas.json", 'w', encoding='utf-8') as f:
    json.dump(decisiones, f, indent=2, ensure_ascii=False)

print("\nDecisiones operativas guardadas en outputs/tables/decisiones_operativas.json")

# BLOQUE B: Reducción de Dimensionalidad

## Objetivo
Este bloque se centra en la aplicación sistemática de diferentes técnicas de reducción de dimensionalidad, utilizando las normalizaciones seleccionadas en el Bloque A.

### Aspectos Clave
1. Consideración cuidadosa de la interacción normalizador ↔ reductor
2. Evaluación de estabilidad para métodos no determinísticos
3. Generación de embeddings reproducibles
4. Análisis de costes computacionales

### Métodos a Implementar
1. **Lineales**:
   - PCA (Principal Component Analysis)
   - NMF (Non-negative Matrix Factorization)
   - SparsePCA
2. **No Lineales**:
   - UMAP
   - t-SNE
   - Isomap
   - Spectral Embedding
3. **Supervisados**:
   - LDA (Linear Discriminant Analysis)
   - Selección de píxeles por MI/RF

In [ ]:
# Importaciones específicas para reducción de dimensionalidad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import gc  # Para gestión de memoria

# Importaciones locales
from src.reduction.reducers import (
    obtener_reductor, ajustar_reductor, transformar_reductor,
    ajustar_transformar_viz, verificar_estabilidad, guardar_embedding,
    guardar_reductor, obtener_autoencoder
)

# Configurar monitores de memoria
import psutil
process = psutil.Process()

# Cargar normalizadores elegidos
with open("../data/processed/normalizadores_elegidos.json", 'r') as f:
    meta_norm = json.load(f)
    
normalizadores_elegidos = meta_norm['elegidos']
print(f"Normalizadores seleccionados: {normalizadores_elegidos}")

# Cargar dimensiones candidatas
df_candidatos = pd.read_csv("../outputs/tables/candidatos_dimension.csv")
grid_dims = sorted(df_candidatos['dimension_candidata'].unique().tolist())

## Pipeline de Reducción

Implementaremos los siguientes métodos de reducción:
1. PCA (Principal Component Analysis)
2. NMF (Non-negative Matrix Factorization)  
3. t-SNE (t-Distributed Stochastic Neighbor Embedding)
4. UMAP (Uniform Manifold Approximation and Projection)
5. Isomap
6. Spectral Embedding

Para cada método:
- Evaluaremos su rendimiento con diferentes dimensiones
- Mediremos tiempo de cómputo
- Analizaremos estabilidad
- Guardaremos embeddings y modelos

In [ ]:
def ejecutar_pipeline_reduccion(X_train, X_test, metodo, normalizador, dims, n_runs=5):
    """
    Ejecuta el pipeline completo de reducción para un método y normalizador específicos.
    
    Args:
        X_train (array): Datos de entrenamiento
        X_test (array): Datos de prueba
        metodo (str): Método de reducción ('pca', 'nmf', etc.)
        normalizador (str): Método de normalización usado
        dims (list): Lista de dimensiones a evaluar
        n_runs (int): Número de corridas para estabilidad
        
    Returns:
        dict: Resultados del pipeline incluyendo tiempos, estabilidad y paths
    """
    resultados = {
        'metodo': metodo,
        'normalizador': normalizador,
        'tiempos': [],
        'memoria': [],
        'estabilidad': [],
        'paths': []
    }
    
    for dim in dims:
        print(f"\nProcesando {metodo} con {dim} dimensiones...")
        
        # 1. Ajustar reductor
        inicio = datetime.now()
        reductor = obtener_reductor(metodo, n_components=dim)
        X_reducido = ajustar_reductor(reductor, X_train)
        tiempo = (datetime.now() - inicio).total_seconds()
        
        # 2. Transformar datos de prueba
        X_test_reducido = transformar_reductor(reductor, X_test)
        
        # 3. Evaluar estabilidad
        estabilidad = verificar_estabilidad(
            metodo, X_train, dim, n_runs=n_runs
        )
        
        # 4. Guardar resultados
        prefijo = f"reducer__{normalizador}__{metodo}__{dim}"
        
        # Guardar embedding
        path_emb = guardar_embedding(
            X_reducido, 
            X_test_reducido,
            prefijo
        )
        
        # Guardar modelo
        path_modelo = guardar_reductor(
            reductor,
            prefijo,
            {
                'tiempo_ajuste': tiempo,
                'memoria_pico': process.memory_info().rss / 1024 / 1024,
                'estabilidad': estabilidad
            }
        )
        
        # Registrar métricas
        resultados['tiempos'].append(tiempo)
        resultados['memoria'].append(process.memory_info().rss / 1024 / 1024)
        resultados['estabilidad'].append(estabilidad)
        resultados['paths'].append({
            'embedding': path_emb,
            'modelo': path_modelo
        })
        
        # Limpiar memoria
        gc.collect()
        
    return resultados

In [ ]:
# Configuración de métodos a evaluar
metodos_reduccion = ['pca', 'nmf', 'tsne', 'umap', 'isomap', 'spectral']

# Resultados globales
resultados_globales = []

# Iterar sobre normalizadores y métodos
for normalizador in normalizadores_elegidos:
    print(f"\nProcesando normalización: {normalizador}")
    
    # Cargar datos normalizados
    X_train = np.load(f"../data/processed/normalizations/{normalizador}/train_X.npy")
    X_test = np.load(f"../data/processed/normalizations/{normalizador}/test_X.npy")
    
    # Ejecutar pipeline para cada método
    for metodo in metodos_reduccion:
        print(f"\nEjecutando {metodo}...")
        
        try:
            resultados = ejecutar_pipeline_reduccion(
                X_train, X_test,
                metodo=metodo,
                normalizador=normalizador,
                dims=grid_dims
            )
            resultados_globales.append(resultados)
            
        except Exception as e:
            print(f"Error en {metodo}: {str(e)}")
            continue
            
        # Limpiar memoria
        gc.collect()

# Convertir resultados a DataFrame
df_resultados = pd.DataFrame(resultados_globales)
df_resultados.to_csv("../outputs/tables/resultados_reduccion.csv", index=False)

## Análisis de Resultados

Analizaremos los resultados de la reducción de dimensionalidad en términos de:
1. Tiempo de cómputo vs. dimensiones
2. Uso de memoria vs. dimensiones  
3. Estabilidad de los embeddings
4. Visualización de embeddings de baja dimensión

In [ ]:
# Cargar resultados
df_resultados = pd.read_csv("../outputs/tables/resultados_reduccion.csv")

# 1. Gráfico de tiempo vs dimensiones
plt.figure(figsize=(12, 6))
for metodo in metodos_reduccion:
    data = df_resultados[df_resultados['metodo'] == metodo]
    plt.plot(grid_dims, data['tiempos'], 'o-', label=metodo)

plt.xlabel('Número de dimensiones')
plt.ylabel('Tiempo (segundos)')
plt.title('Tiempo de cómputo vs Dimensiones')
plt.legend()
plt.grid(True)
plt.savefig('../outputs/figures/tiempo_vs_dims.png')
plt.close()

# 2. Gráfico de memoria vs dimensiones
plt.figure(figsize=(12, 6))
for metodo in metodos_reduccion:
    data = df_resultados[df_resultados['metodo'] == metodo]
    plt.plot(grid_dims, data['memoria'], 'o-', label=metodo)

plt.xlabel('Número de dimensiones')
plt.ylabel('Memoria (MB)')
plt.title('Uso de Memoria vs Dimensiones')
plt.legend()
plt.grid(True)
plt.savefig('../outputs/figures/memoria_vs_dims.png')
plt.close()

# 3. Gráfico de estabilidad
plt.figure(figsize=(12, 6))
for metodo in metodos_reduccion:
    data = df_resultados[df_resultados['metodo'] == metodo]
    plt.plot(grid_dims, data['estabilidad'], 'o-', label=metodo)

plt.xlabel('Número de dimensiones')
plt.ylabel('Estabilidad (correlación)')
plt.title('Estabilidad vs Dimensiones')
plt.legend()
plt.grid(True)
plt.savefig('../outputs/figures/estabilidad_vs_dims.png')
plt.close()

# Tabla resumen
tabla_resumen = df_resultados.groupby('metodo').agg({
    'tiempos': ['mean', 'std'],
    'memoria': ['mean', 'std'],
    'estabilidad': ['mean', 'std']
}).round(3)

print("\nResumen de métricas por método:")
print(tabla_resumen)

In [ ]:
# Cargar etiquetas
y_train = np.load("../data/raw/train_Y.npy")

# Visualizar embeddings 2D para cada método
for metodo in metodos_reduccion:
    print(f"\nVisualizando embeddings 2D para {metodo}...")
    try:
        # Cargar embedding 2D
        X_reducido = np.load(f"../data/processed/embeddings/reducer__clahe__{metodo}__2/train_X.npy")
        
        # Crear scatter plot
        plt.figure(figsize=(10, 10))
        scatter = plt.scatter(X_reducido[:, 0], X_reducido[:, 1], 
                            c=y_train, cmap='tab10', alpha=0.6)
        plt.colorbar(scatter)
        
        plt.title(f'Embedding 2D usando {metodo}')
        plt.xlabel('Componente 1')
        plt.ylabel('Componente 2')
        plt.grid(True)
        
        plt.savefig(f'../outputs/figures/embedding_2d_{metodo}.png')
        plt.close()
        
    except Exception as e:
        print(f"Error visualizando {metodo}: {str(e)}")
        continue

# BLOQUE C: Clasificación y Evaluación

En este bloque evaluaremos el rendimiento de diferentes clasificadores sobre los embeddings generados:

1. Clasificadores a evaluar:
   - SVM (Support Vector Machine)
   - Random Forest
   - KNN (K-Nearest Neighbors)
   - Regresión Logística

2. Métricas de evaluación:
   - Accuracy
   - Precision por clase
   - Recall por clase
   - F1-score por clase
   - Matriz de confusión

3. Análisis de resultados:
   - Comparación entre métodos
   - Impacto de la dimensionalidad
   - Tiempo de entrenamiento/inferencia

In [ ]:
# Importaciones para clasificación
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

# Definir clasificadores
clasificadores = {
    'svm': SVC(random_state=42),
    'rf': RandomForestClassifier(n_estimators=100, random_state=42),
    'knn': KNeighborsClassifier(n_neighbors=5),
    'logreg': LogisticRegression(random_state=42)
}

# Cargar etiquetas
y_train = np.load("../data/raw/train_Y.npy")
y_test = np.load("../data/raw/test_Y.npy")

# Lista para almacenar resultados
resultados_clasificacion = []

# Iterar sobre métodos de reducción y dimensiones
for metodo in metodos_reduccion:
    for dim in grid_dims:
        print(f"\nProcesando {metodo} con {dim} dimensiones...")
        
        try:
            # Cargar embeddings
            X_train = np.load(f"../data/processed/embeddings/reducer__clahe__{metodo}__{dim}/train_X.npy")
            X_test = np.load(f"../data/processed/embeddings/reducer__clahe__{metodo}__{dim}/test_X.npy")
            
            # Evaluar cada clasificador
            for nombre_clf, clf in clasificadores.items():
                print(f"Evaluando {nombre_clf}...")
                
                # Entrenar y medir tiempo
                inicio = datetime.now()
                clf.fit(X_train, y_train)
                tiempo_train = (datetime.now() - inicio).total_seconds()
                
                # Predecir y medir tiempo
                inicio = datetime.now()
                y_pred = clf.predict(X_test)
                tiempo_pred = (datetime.now() - inicio).total_seconds()
                
                # Calcular métricas
                accuracy = accuracy_score(y_test, y_pred)
                precision, recall, f1, _ = precision_recall_fscore_support(
                    y_test, y_pred, average='weighted'
                )
                
                # Guardar resultados
                resultados_clasificacion.append({
                    'metodo_reduccion': metodo,
                    'dimensiones': dim,
                    'clasificador': nombre_clf,
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'tiempo_train': tiempo_train,
                    'tiempo_pred': tiempo_pred
                })
                
                # Guardar matriz de confusión
                cm = confusion_matrix(y_test, y_pred)
                plt.figure(figsize=(10, 8))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
                plt.title(f'Matriz de Confusión - {metodo}_{dim}d_{nombre_clf}')
                plt.xlabel('Predicción')
                plt.ylabel('Real')
                plt.savefig(f'../outputs/figures/confusion_{metodo}_{dim}d_{nombre_clf}.png')
                plt.close()
                
        except Exception as e:
            print(f"Error: {str(e)}")
            continue
            
# Convertir resultados a DataFrame
df_clasificacion = pd.DataFrame(resultados_clasificacion)
df_clasificacion.to_csv("../outputs/tables/resultados_clasificacion.csv", index=False)

## Análisis de Resultados de Clasificación

Analizaremos el rendimiento de los clasificadores considerando:
1. Accuracy vs dimensiones para cada método/clasificador
2. Tiempo de entrenamiento/predicción
3. Balance precisión-recall
4. Mejores combinaciones método-clasificador

In [ ]:
# Cargar resultados
df_clasificacion = pd.read_csv("../outputs/tables/resultados_clasificacion.csv")

# 1. Gráfico de accuracy vs dimensiones
plt.figure(figsize=(15, 10))
for metodo in metodos_reduccion:
    for clf in clasificadores.keys():
        data = df_clasificacion[
            (df_clasificacion['metodo_reduccion'] == metodo) & 
            (df_clasificacion['clasificador'] == clf)
        ]
        plt.plot(data['dimensiones'], data['accuracy'], 
                'o-', label=f'{metodo}_{clf}')

plt.xlabel('Número de dimensiones')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Dimensiones por Método y Clasificador')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('../outputs/figures/accuracy_vs_dims.png')
plt.close()

# 2. Gráfico de tiempo de entrenamiento
plt.figure(figsize=(15, 10))
for metodo in metodos_reduccion:
    for clf in clasificadores.keys():
        data = df_clasificacion[
            (df_clasificacion['metodo_reduccion'] == metodo) & 
            (df_clasificacion['clasificador'] == clf)
        ]
        plt.plot(data['dimensiones'], data['tiempo_train'], 
                'o-', label=f'{metodo}_{clf}')

plt.xlabel('Número de dimensiones')
plt.ylabel('Tiempo de entrenamiento (s)')
plt.title('Tiempo de Entrenamiento vs Dimensiones')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('../outputs/figures/tiempo_train_vs_dims.png')
plt.close()

# 3. Gráfico precision vs recall
plt.figure(figsize=(10, 10))
for metodo in metodos_reduccion:
    for clf in clasificadores.keys():
        data = df_clasificacion[
            (df_clasificacion['metodo_reduccion'] == metodo) & 
            (df_clasificacion['clasificador'] == clf)
        ]
        plt.scatter(data['precision'], data['recall'], 
                   label=f'{metodo}_{clf}', alpha=0.6)

plt.xlabel('Precision')
plt.ylabel('Recall')
plt.title('Precision vs Recall por Método y Clasificador')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.savefig('../outputs/figures/precision_vs_recall.png')
plt.close()

# 4. Tabla de mejores combinaciones
mejores = df_clasificacion.sort_values('f1', ascending=False).head(10)
print("\nTop 10 mejores combinaciones método-clasificador:")
print(mejores[['metodo_reduccion', 'dimensiones', 'clasificador', 'accuracy', 'f1']])

# Conclusiones y Recomendaciones

## Resumen de Hallazgos

1. **Métodos de Reducción**:
   - Rendimiento por método (tiempo/memoria)
   - Estabilidad de los embeddings
   - Preservación de estructura de datos

2. **Clasificación**:
   - Mejores combinaciones método-clasificador
   - Balance dimensionalidad vs rendimiento
   - Tiempos de entrenamiento/inferencia

3. **Recomendaciones Prácticas**:
   - Selección de método según caso de uso
   - Dimensionalidad óptima
   - Consideraciones de escalabilidad

In [ ]:
# 1. Análisis de métodos de reducción
print("=== Análisis de Métodos de Reducción ===")

# Resumen por método
resumen_metodos = df_resultados.groupby('metodo').agg({
    'tiempos': ['mean', 'std'],
    'memoria': ['mean', 'std'],
    'estabilidad': ['mean', 'std']
}).round(3)

print("\nMétricas promedio por método:")
print(resumen_metodos)

# Identificar mejor método por métrica
mejor_tiempo = df_resultados.groupby('metodo')['tiempos'].mean().idxmin()
mejor_memoria = df_resultados.groupby('metodo')['memoria'].mean().idxmin()
mejor_estabilidad = df_resultados.groupby('metodo')['estabilidad'].mean().idxmax()

print(f"\nMejor método por tiempo: {mejor_tiempo}")
print(f"Mejor método por memoria: {mejor_memoria}")
print(f"Mejor método por estabilidad: {mejor_estabilidad}")

# 2. Análisis de clasificación
print("\n=== Análisis de Clasificación ===")

# Resumen por combinación método-clasificador
resumen_clf = df_clasificacion.groupby(['metodo_reduccion', 'clasificador']).agg({
    'accuracy': ['mean', 'std'],
    'f1': ['mean', 'std'],
    'tiempo_train': ['mean', 'std']
}).round(3)

print("\nMétricas promedio por combinación método-clasificador:")
print(resumen_clf)

# Identificar mejores combinaciones
mejor_accuracy = df_clasificacion.loc[df_clasificacion['accuracy'].idxmax()]
mejor_f1 = df_clasificacion.loc[df_clasificacion['f1'].idxmax()]

print(f"\nMejor combinación por accuracy:")
print(f"Método: {mejor_accuracy['metodo_reduccion']}")
print(f"Clasificador: {mejor_accuracy['clasificador']}")
print(f"Dimensiones: {mejor_accuracy['dimensiones']}")
print(f"Accuracy: {mejor_accuracy['accuracy']:.3f}")

print(f"\nMejor combinación por F1:")
print(f"Método: {mejor_f1['metodo_reduccion']}")
print(f"Clasificador: {mejor_f1['clasificador']}")
print(f"Dimensiones: {mejor_f1['dimensiones']}")
print(f"F1: {mejor_f1['f1']:.3f}")

# 3. Análisis de dimensionalidad óptima
print("\n=== Análisis de Dimensionalidad Óptima ===")

# Calcular punto de inflexión en accuracy vs dimensiones
for metodo in metodos_reduccion:
    for clf in clasificadores.keys():
        data = df_clasificacion[
            (df_clasificacion['metodo_reduccion'] == metodo) & 
            (df_clasificacion['clasificador'] == clf)
        ]
        
        # Encontrar dimensión donde accuracy se estabiliza
        acc_diff = data['accuracy'].diff()
        dim_opt = data.loc[
            (acc_diff < 0.01) & (acc_diff > -0.01), 
            'dimensiones'
        ].iloc[0] if len(acc_diff) > 0 else None
        
        if dim_opt:
            print(f"\n{metodo} + {clf}:")
            print(f"Dimensionalidad óptima: {dim_opt}")
            print(f"Accuracy en ese punto: {data[data['dimensiones'] == dim_opt]['accuracy'].values[0]:.3f}")

# Guardar conclusiones
conclusiones = {
    'mejor_metodo_tiempo': mejor_tiempo,
    'mejor_metodo_memoria': mejor_memoria,
    'mejor_metodo_estabilidad': mejor_estabilidad,
    'mejor_combinacion_accuracy': {
        'metodo': mejor_accuracy['metodo_reduccion'],
        'clasificador': mejor_accuracy['clasificador'],
        'dimensiones': int(mejor_accuracy['dimensiones']),
        'accuracy': float(mejor_accuracy['accuracy'])
    },
    'mejor_combinacion_f1': {
        'metodo': mejor_f1['metodo_reduccion'],
        'clasificador': mejor_f1['clasificador'],
        'dimensiones': int(mejor_f1['dimensiones']),
        'f1': float(mejor_f1['f1'])
    }
}

with open('../outputs/tables/conclusiones.json', 'w') as f:
    json.dump(conclusiones, f, indent=2)

## Recomendaciones Finales

1. **Selección de Método de Reducción**:
   - Para velocidad: Usar el método más rápido identificado
   - Para estabilidad: Preferir el método más estable
   - Para memoria limitada: Elegir el método más eficiente en memoria

2. **Dimensionalidad**:
   - Usar el punto de inflexión identificado como guía
   - Considerar el trade-off entre compresión y rendimiento
   - Validar con datos específicos del dominio

3. **Clasificación**:
   - Elegir el clasificador según métricas prioritarias
   - Considerar tiempo de entrenamiento vs accuracy
   - Validar con cross-validation adicional si es necesario

4. **Consideraciones Prácticas**:
   - Monitorear uso de recursos en producción
   - Implementar pipeline de reentrenamiento si necesario
   - Documentar parámetros óptimos encontrados